# Draft Audit -- Full Demo (Layer 1 + Layer 2 + Layer 3)

This notebook runs on **your own free Google account**. Nothing here is shared with anyone else's session, and Layer 1 + Layer 2 cost you nothing.

**Before running:** go to `Runtime` -> `Change runtime type` -> set Hardware accelerator to **T4 GPU** -> Save. Then run the cells below in order (Shift+Enter on each).

What each layer needs:
- **Layer 1** (pattern rules): instant, no GPU, no cost.
- **Layer 2** (statistical scorer, Binoculars): needs the T4 GPU; downloads about 3GB of model weights on first use (a few minutes -- that's normal, not a hang). Free.
- **Layer 3** (adversarial robustness check): uses the Claude API, so it needs your own [Anthropic API key](https://console.anthropic.com) and costs a small amount per call. Not free. See the Layer 3 section for setup.

You can skip Layer 3 entirely if you don't want to set up an API key -- Layers 1 and 2 stand on their own.

See the project's findings docs -- [FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/FINDINGS.md), [LAYER2_FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/LAYER2_FINDINGS.md), [HUMANIZER_FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/HUMANIZER_FINDINGS.md), and [LAYER3_FINDINGS.md](https://github.com/Gh12890/ai-writing-detector/blob/master/LAYER3_FINDINGS.md) -- for what these numbers actually mean and their real, documented limitations. Nothing in this notebook is a validated, calibrated "this is AI" verdict.

## 1. Get the code

Clones the public repo directly -- no manual uploads needed. Note: `corpus/` and `ai_corpus/` are intentionally excluded from the repo (private, consent-gated documents), so cloning does *not* give you access to any of the project's real test documents. You'll paste in your own text below.

In [ ]:
!git clone https://github.com/Gh12890/ai-writing-detector.git
%cd ai-writing-detector
!pip install -q -r requirements.txt

## 2. Paste the text you want to analyze

Edit the text between the triple quotes below, then run this cell.

In [ ]:
text = """
Paste your text here, replacing this placeholder.
""".strip()

print(f"Loaded {len(text.split())} words.")

## 3. Layer 1 -- pattern rules (instant, no GPU needed)

In [ ]:
from layer1 import PatternScorer

result = PatternScorer().analyze(text)

print(f"Word count: {result.word_count}")
print(f"Total flags: {result.flag_count}")
print(f"Structural /1000w: {result.structural_density_per_1000w}")
print(f"Lexical /1000w: {result.lexical_density_per_1000w}")
if result.suppressed_count:
    print(f"{result.suppressed_count} flag(s) suppressed as legal boilerplate.")

by_tier = result.flags_by_tier()
print(f"\nStructural flags ({len(by_tier['structural'])}):")
for f in by_tier["structural"]:
    print(f"  [{f.rule_id}] {f.match_text[:60]!r}")
print(f"\nLexical flags ({len(by_tier['lexical'])}):")
for f in by_tier["lexical"]:
    print(f"  [{f.rule_id}] {f.match_text[:60]!r}")

## 4. Layer 2 -- statistical scorer (Binoculars)

Downloads ~3GB of model weights on first run. Takes a few minutes the first time; fast after that within the same session.

In [ ]:
from layer2_binoculars import load_models, binoculars_score, binoculars_score_full_document
from IPython.display import Markdown, display

print("Loading models (first run downloads ~3GB)...")
observer, performer, tok, device = load_models()
print(f"Loaded. Running on: {device}")

truncated_score = binoculars_score(text, observer, performer, tok, device)
full_result = binoculars_score_full_document(text, observer, performer, tok, device)

print(f"\nTruncated score (first ~512 tokens): {truncated_score:.4f}")
if full_result["pooled_score"] is not None:
    print(f"Full-document pooled score: {full_result['pooled_score']:.4f} "
          f"({full_result['num_chunks']} chunks)")
print("\nCloser to 1.0 = more machine-like, further = more human-like, per the "
      "Binoculars paper's convention. Neither score is validated or calibrated -- "
      "there is no threshold that means 'this is AI.' See LAYER2_FINDINGS.md.")

# Plain-language lean, in bold via Markdown (plain print() can't render bold).
# Anchors measured directly from this project's own corpus (11 documents,
# 2026-08-14): human mean distance 0.112 (excludes sample05, a documented
# outlier at 0.602 -- informal WhatsApp-style text, ~5x any other document),
# AI mean distance 0.125. These two means are close, and the human/AI
# distance RANGES overlap almost completely (human 0.035-0.195 excl.
# outlier, AI 0.063-0.203) -- this is a real finding, not a modeling
# choice: this project's data does not show clean separation by this
# metric. The lean below is intentionally hedged, not a strength claim.
HUMAN_MEAN_DISTANCE = 0.112
AI_MEAN_DISTANCE = 0.125
MIDPOINT = (HUMAN_MEAN_DISTANCE + AI_MEAN_DISTANCE) / 2
MARGIN = 0.02

def score_lean(distance):
    if distance < MIDPOINT - MARGIN:
        return "ai"
    elif distance > MIDPOINT + MARGIN:
        return "human"
    else:
        return "unclear"

LEAN_TEXT = {
    "ai": "**This document's writing statistics sit closer to this project's typical AI-document scores.**",
    "human": "**This document's writing statistics sit closer to this project's typical human-document scores.**",
    "unclear": "**This document sits in the range where this project's human and AI documents overlap -- no reliable lean either way.**",
}

trunc_lean = score_lean(abs(truncated_score - 1.0))
display(Markdown(LEAN_TEXT[trunc_lean]))

if full_result["pooled_score"] is not None:
    pooled_lean = score_lean(abs(full_result["pooled_score"] - 1.0))
    if pooled_lean != trunc_lean:
        display(Markdown(
            f"**Heads up: the two scoring methods disagree on this document** -- "
            f"one leans '{trunc_lean}', the other leans '{pooled_lean}'."
        ))

display(Markdown(
    "This project's own measured human and AI score ranges overlap heavily "
    "(11 documents total, see LAYER2_FINDINGS.md) -- treat any lean above as "
    "a weak hint, not a verdict, and never as a substitute for Layer 1 or "
    "Layer 3's results."
))

## 5. Layer 3 -- adversarial robustness check (Claude API)

**Not a tool to help evade detection.** This answers one question honestly: if this text is flagged, how much does that verdict survive a paraphrase attack?

This cell uses the **Claude API**, not a local model -- you'll need your own Anthropic API key. This costs a small amount of money per call (unlike everything else in this notebook, which is free), and sends your pasted text to a third-party API. See LAYER3_FINDINGS.md for why: a local free model (Qwen2.5-1.5B) was tried first and reliably compressed whole documents into summaries instead of paraphrasing them, even after a chunking workaround -- the Claude API preserves length directly with no such workaround needed.

On this project's own test data (5 documents total, train/dev + held-out), the Claude API attack never once reduced detectability, and made the text more detectable on 3 of 5 -- a real, if still small-sample, finding, not a guarantee either way. Treat any single result below as one data point, not a reliable prediction.

**Setup, one-time:** get a key at [console.anthropic.com](https://console.anthropic.com), then in Colab click the key icon in the left sidebar -> Secrets -> add `ANTHROPIC_API_KEY` -> toggle Notebook access on. Do not paste your key directly into a cell.

In [ ]:
!pip install -q anthropic

from google.colab import userdata
import os
import anthropic

try:
    api_key = userdata.get("ANTHROPIC_API_KEY")
except userdata.SecretNotFoundError:
    api_key = None

if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Click the key icon in the left sidebar, "
        "go to Secrets, add ANTHROPIC_API_KEY with your key as the value, "
        "and toggle 'Notebook access' on for this notebook. Then re-run this cell. "
        "Skip this and Layer 3 if you don't want to set up an API key -- Layers 1 and 2 don't need it."
    )

os.environ["ANTHROPIC_API_KEY"] = api_key
client = anthropic.Anthropic(api_key=api_key)
print("Client ready.")

In [ ]:
from layer3_adversarial import run_adversarial_test

print("Generating paraphrase via Claude API...")
l3_result = run_adversarial_test(text, client)

print(f"\nOriginal:   {l3_result['original_word_count']} words, {l3_result['original_flags']} flags")
print(f"  Rules: {l3_result['original_rules']}")
print(f"Paraphrase: {l3_result['paraphrase_word_count']} words, {l3_result['paraphrase_flags']} flags")
print(f"  Rules: {l3_result['paraphrase_rules']}")
print(f"Word count ratio: {l3_result['word_count_ratio']:.2f}")

print("\n--- Paraphrased text (for reference only) ---\n")
print(l3_result["paraphrase_text"])

---
Questions about the methodology, the honest limitations, or the real measured results behind any number above: see the four findings documents in the [repo](https://github.com/Gh12890/ai-writing-detector) -- `FINDINGS.md`, `LAYER2_FINDINGS.md`, `HUMANIZER_FINDINGS.md`, `LAYER3_FINDINGS.md`.